In [1]:
"""
One SADCP figure per eddy, three panels:
  (a) map: ADT, TOEddies contours, ship track, eddy center
  (b) velocity vs distance at a few depths
  (c) velocity section (distance, depth)

Two strategies:
  "section" (cyclone CE18, crossed through the center): velocity orthogonal to the
      ship track, v_perp, against distance along the section.
  "radial"  (anticyclone AE11, never crossed): azimuthal velocity v_theta in the frame
      moving with the eddy center, against distance to the center.
"""
import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
if not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.lines as mlines
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean

try:
    import geopandas as gpd
except ImportError:
    gpd = None


In [2]:

CMAP = cmocean.cm.balance
crs = ccrs.PlateCarree()

# =============================================================================
# Eddies
# =============================================================================
EDDIES = {
    "CE18": dict(name="CE18_20260523", eddy_type=-1, mode="section",
                 t0="2026-07-16T06:00", t1="2026-07-17T16:00"),
    "AE11": dict(name="AE11_20260514", eddy_type=1, mode="radial",
                 t0="2026-07-01T00:00", t1="2026-07-11T00:00", r_limit_km=220),
}
RUN = ["CE18", "AE11"]

# =============================================================================
# Settings
# =============================================================================
SADCP_AVG = "10min"          # SADCP time averaging (~3 km at 10 kn)
USE_DETIDED = True           # use UVEL_ADCP_CORTIDE / VVEL_ADCP_CORTIDE when present
Z_SPLIT = 150.0              # m: OS150 above, OS38 below
ZMAX_SADCP = 1000.0
Z_FILL_GAP = 40.0            # m: fill vertical gaps up to this size (seam between OS150/OS38)
DS_KM = 3.0                  # bin size along a section
R_BIN_KM = 10.0              # radius bin (radial mode)
FIT_CENTER = True            # radial mode: refine the TOEddies center from SADCP
FIT_LAYER = (30.0, 200.0)    # m: layer used to fit the center
FIT_SEARCH_KM = 60.0         # search +- this offset around the TOEddies center
CENTER_SMOOTH_DAYS = 3       # running mean on the daily TOEddies centers
PROFILE_DEPTHS = [50, 100, 200, 400, 700]
R_KM = 6371.0

# =============================================================================
# Paths
# =============================================================================
BASE = Path("/Users/emmagurcan/Documents/France/ENS/Speich/WHIRLS")
DIR_SADCP = BASE / "data/SADCP"
SADCP_FILES = {"OS150": DIR_SADCP / "WHIRLS_NB150_TOT_osite.nc",
               "OS38": DIR_SADCP / "WHIRLS_BB38_TOT_osite.nc"}
DIR_SSH = BASE / "data/SSH"
DIR_EDDIES = {"TOEDDIES_better_ADT": BASE / "data/TOEDDIES/TOEDDIES_better_ADT",
              "TOEDDIES": BASE / "data/TOEDDIES/TOEDDIES"}
DIR_TRACKS = BASE / "figures"
OUT_DIR = BASE / "figures2"


# =============================================================================
# Geometry
# =============================================================================
def to_xy(lon, lat, lon0, lat0):
    """Local tangent plane, km."""
    x = np.deg2rad(np.asarray(lon, float) - lon0) * R_KM * np.cos(np.deg2rad(lat0))
    y = np.deg2rad(np.asarray(lat, float) - lat0) * R_KM
    return x, y


def to_lonlat(x, y, lon0, lat0):
    lon = lon0 + np.rad2deg(np.asarray(x, float) / (R_KM * np.cos(np.deg2rad(lat0))))
    lat = lat0 + np.rad2deg(np.asarray(y, float) / R_KM)
    return lon, lat


def fit_line(x, y):
    """Principal axis of the points: origin, along unit vector (direction of travel),
    normal unit vector (LEFT of travel)."""
    ok = np.isfinite(x) & np.isfinite(y)
    xm, ym = x[ok].mean(), y[ok].mean()
    w, vec = np.linalg.eigh(np.cov(np.vstack([x[ok] - xm, y[ok] - ym])))
    es = vec[:, np.argmax(w)]
    xi, yi = x[ok][[0, -1]], y[ok][[0, -1]]
    if (xi[1] - xi[0]) * es[0] + (yi[1] - yi[0]) * es[1] < 0:
        es = -es
    return np.array([xm, ym]), es, np.array([-es[1], es[0]])


def polar(x, y, u, v):
    """Radius (km), azimuthal (+ anticlockwise) and radial (+ outward) velocity."""
    r = np.hypot(x, y)
    rr = np.where(r > 0, r, np.nan)
    if np.ndim(u) == 2:
        x, y, rr = x[:, None], y[:, None], rr[:, None]
    return r, (-u * y + v * x) / rr, (u * x + v * y) / rr


def fit_center(x, y, u, v, r_max, search=FIT_SEARCH_KM, step=2.0, r_min=10.0, nmin=20):
    """Offset (dx, dy) of the center minimising the fraction of radial kinetic energy."""
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(u) & np.isfinite(v)
    x, y, u, v = x[ok], y[ok], u[ok], v[ok]
    best = (np.inf, 0.0, 0.0)
    for dx in np.arange(-search, search + step / 2, step):
        for dy in np.arange(-search, search + step / 2, step):
            r, vt, vr = polar(x - dx, y - dy, u, v)
            m = (r >= r_min) & (r <= r_max)
            if m.sum() < nmin:
                continue
            cost = np.sum(vr[m] ** 2) / np.sum(vr[m] ** 2 + vt[m] ** 2)
            if cost < best[0]:
                best = (cost, dx, dy)
    return best[1], best[2], best[0]


def bin_columns(coord, vals, edges, nmin=1):
    """Mean of vals (n, nz) in bins of coord. Returns centers, (nbins, nz) mean, counts."""
    idx = np.digitize(coord, edges) - 1
    out = np.full((len(edges) - 1, vals.shape[1]), np.nan)
    cnt = np.zeros(len(edges) - 1, int)
    for b in range(len(edges) - 1):
        m = idx == b
        cnt[b] = m.sum()
        if cnt[b] >= nmin:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                out[b] = np.nanmean(vals[m], axis=0)
    return 0.5 * (edges[:-1] + edges[1:]), out, cnt


def fill_z_gaps(V, z, max_gap=Z_FILL_GAP):
    """Linear interpolation in depth across interior gaps no thicker than max_gap."""
    V = V.copy()
    for i in range(V.shape[0]):
        ok = np.isfinite(V[i])
        if ok.sum() < 2:
            continue
        zk = z[ok]
        bad = np.where(~ok & (z > zk.min()) & (z < zk.max()))[0]
        if not len(bad):
            continue
        j = np.searchsorted(zk, z[bad])
        small = (zk[j] - zk[j - 1]) <= max_gap
        V[i, bad[small]] = np.interp(z[bad[small]], zk, V[i, ok])
    return V


# =============================================================================
# Data loading
# =============================================================================
def _find(ds, names, required=True):
    for n in names:
        if n in ds.variables:
            return n
    low = {k.lower(): k for k in ds.variables}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    if required:
        raise KeyError(f"None of {names} in {list(ds.variables)}")
    return None


_SADCP_CACHE = {}


def load_sadcp_raw(path):
    """Full-resolution SADCP as Dataset(time, z) with u, v, lat, lon (QC flag == 1)."""
    if path in _SADCP_CACHE:
        return _SADCP_CACHE[path]
    ds = xr.open_dataset(path)
    tn = _find(ds, ["JULD_ADCP", "TIME", "time", "JULD"])
    zn = _find(ds, ["DEPH", "DEPTH", "depth"])
    un = _find(ds, ["UVEL_ADCP_CORTIDE"], required=False) if USE_DETIDED else None
    vn = _find(ds, ["VVEL_ADCP_CORTIDE"], required=False) if USE_DETIDED else None
    if un is None or vn is None:
        un, vn = _find(ds, ["UVEL_ADCP"]), _find(ds, ["VVEL_ADCP"])
    fn = _find(ds, ["CAS_CURRENT_FLAG"], required=False)
    print(f"  {Path(path).name}: using {un}, {vn}")
    tdim = ds[tn].dims[0]
    U = ds[un].squeeze(drop=True)
    zdim = [d for d in U.dims if d != tdim][0]
    u = U.transpose(tdim, zdim).values.astype(float)
    v = ds[vn].squeeze(drop=True).transpose(tdim, zdim).values.astype(float)
    traw = ds[tn].values
    time = (pd.to_datetime(traw) if np.issubdtype(traw.dtype, np.datetime64)
            else pd.Timestamp("1950-01-01") + pd.to_timedelta(traw.astype(float), unit="D"))
    if fn is not None:
        fl = ds[fn].squeeze(drop=True)
        fl = fl.transpose(tdim, zdim).values if zdim in fl.dims else fl.values[:, None]
        good = (fl == 1) if np.issubdtype(fl.dtype, np.number) else \
            np.char.strip(np.char.decode(fl.astype("S"), "ascii", "ignore")) == "1"
        good = np.broadcast_to(good, u.shape)
        u, v = np.where(good, u, np.nan), np.where(good, v, np.nan)
    u[np.abs(u) > 5] = np.nan
    v[np.abs(v) > 5] = np.nan
    zda = ds[zn].squeeze(drop=True)
    z = np.abs(np.nanmedian(zda.transpose(tdim, zdim).values, axis=0) if zda.ndim == 2
               else zda.values).astype(float)
    keepz = np.isfinite(z)
    lat = ds[_find(ds, ["LATITUDE", "lat"])].squeeze(drop=True).values.astype(float)
    lon = ds[_find(ds, ["LONGITUDE", "lon"])].squeeze(drop=True).values.astype(float)
    lat = np.where(np.abs(lat) <= 90, lat, np.nan)
    lon = np.where(np.abs(lon) <= 360, lon, np.nan)
    lon = np.where(lon > 180, lon - 360, lon)
    out = xr.Dataset({"u": (("time", "z"), u[:, keepz]), "v": (("time", "z"), v[:, keepz]),
                      "lat": ("time", lat), "lon": ("time", lon)},
                     coords={"time": np.asarray(time, "datetime64[ns]"), "z": z[keepz]})
    out = out.isel(time=np.asarray(pd.notna(time))).sortby("time").sortby("z")
    out = out.isel(time=~pd.Index(out.time.values).duplicated())
    ds.close()
    _SADCP_CACHE[path] = out
    return out


def sadcp_window(t0, t1):
    """OS150 above Z_SPLIT + OS38 below, averaged over SADCP_AVG, inside [t0, t1].
    Returns a dict of numpy arrays: time, lon, lat, z, u (nt, nz), v (nt, nz)."""
    parts = []
    for inst, f in SADCP_FILES.items():
        if not f.exists():
            print(f"  ! missing {f}")
            continue
        d = load_sadcp_raw(f).sel(time=slice(t0, t1)).resample(time=SADCP_AVG).mean()
        zz = d.z.values
        keep = zz <= Z_SPLIT if inst == "OS150" else (zz > Z_SPLIT) & (zz <= ZMAX_SADCP)
        parts.append(d.isel(z=np.where(keep)[0]))
    if not parts:
        raise FileNotFoundError("no SADCP file found")
    out = parts[0]
    for p in parts[1:]:
        p = p.reindex(time=out.time)
        out = xr.Dataset({"u": xr.concat([out.u, p.u], "z"), "v": xr.concat([out.v, p.v], "z"),
                          "lat": out.lat.fillna(p.lat), "lon": out.lon.fillna(p.lon)})
    out = out.sortby("z")
    ok = np.isfinite(out.lat.values) & np.isfinite(out.lon.values) & np.isfinite(out.u.values).any(1)
    out = out.isel(time=ok)
    return dict(time=out.time.values, lon=out.lon.values, lat=out.lat.values, z=out.z.values,
                u=out.u.values, v=out.v.values)


def load_track(name):
    f = next(iter(sorted(DIR_TRACKS.rglob(f"tracked_eddy_{name}.csv"))), None)
    if f is None:
        raise FileNotFoundError(f"tracked_eddy_{name}.csv not found under {DIR_TRACKS}")
    return pd.read_csv(f, parse_dates=["date"])


def center_at(tr, times):
    """TOEddies center at given times: daily centers at 12:00, running mean, linear interp."""
    t = tr.date + pd.Timedelta(hours=12)
    lon = tr.lon.rolling(CENTER_SMOOTH_DAYS, center=True, min_periods=1).mean().values
    lat = tr.lat.rolling(CENTER_SMOOTH_DAYS, center=True, min_periods=1).mean().values
    tq = pd.to_datetime(times).values.astype("datetime64[s]").astype(float)
    tt = t.values.astype("datetime64[s]").astype(float)
    return np.interp(tq, tt, lon), np.interp(tq, tt, lat)


def rmax_at(tr, t0, t1):
    m = (tr.date >= pd.Timestamp(t0).normalize()) & (tr.date <= pd.Timestamp(t1))
    return float(tr.loc[m, "r_maxspeed_km"].mean())


def eddy_contours(tr, day, eddy_type):
    """Max-speed and outermost contours of the tracked eddy on that day (lon/lat arrays)."""
    if gpd is None:
        return None, None
    row = tr.iloc[(tr.date - pd.Timestamp(day).normalize()).abs().argmin()]
    folder = DIR_EDDIES.get(row.source, DIR_EDDIES["TOEDDIES"])
    kind = "anticyclones" if eddy_type == 1 else "cyclones"
    d = row.date
    res = []
    for part in ("maxspeed", "outcontour"):
        xy = None
        for tag in (d.strftime("%Y%m%d"), d.strftime("%Y-%m-%d")):
            for f in sorted(glob.glob(str(folder / f"*{tag}*{part}*"))):
                if (kind == "anticyclones") != ("anticyclon" in Path(f).name.lower()):
                    continue
                g = gpd.read_file(f).geometry
                i = int(row.index_in_file)
                if i < len(g) and g.iloc[i] is not None and not g.iloc[i].is_empty:
                    geom = g.iloc[i]
                    if geom.geom_type.startswith("Multi"):
                        geom = max(geom.geoms, key=lambda q: q.length)
                    c = geom.exterior.coords if geom.geom_type == "Polygon" else geom.coords
                    xy = np.asarray(c)[:, :2]
                break
            if xy is not None:
                break
        res.append(xy)
    return res


def adt_day(day):
    d = pd.Timestamp(day)
    tags = (d.strftime("%Y-%m-%d"), d.strftime("%Y%m%d"))
    f = next((f for f in sorted(glob.glob(str(DIR_SSH / "*.nc")))
              if any(t in Path(f).name for t in tags)), None)
    if f is None:
        return None
    ds = xr.open_dataset(f)
    ds = ds.rename({k: v for k, v in {"latitude": "lat", "longitude": "lon"}.items()
                    if k in ds.variables})
    a = ds["adt"]
    if "time" in a.dims:
        a = a.isel(time=0)
    return a.load()


# =============================================================================
# Plot helpers
# =============================================================================
def vlim_of(*arrs):
    v = np.concatenate([np.abs(np.asarray(a)).ravel() for a in arrs])
    v = v[np.isfinite(v)]
    return max(0.1, np.round(np.nanpercentile(v, 98) * 10) / 10) if v.size else 0.5


def eddy_color(cfg):
    return "tab:red" if cfg["eddy_type"] == 1 else "tab:blue"


def geo_map(ax, extent, tr, day, eddy_type):
    ax.set_extent(extent, crs=crs)
    ax.add_feature(cfeature.LAND, facecolor="0.6", edgecolor="k", zorder=3)
    gl = ax.gridlines(draw_labels=True, lw=0.3, color="0.5", alpha=0.5)
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = gl.ylabel_style = {"size": 7}
    a = adt_day(day)
    if a is not None:
        a = a.sel(lon=slice(extent[0] - 1, extent[1] + 1), lat=slice(extent[2] - 1, extent[3] + 1))
        ax.contour(a.lon, a.lat, a, levels=np.arange(-2, 3, 0.05), colors="0.5",
                   linewidths=0.4, transform=crs, zorder=1)
    ms, oc = eddy_contours(tr, day, eddy_type)
    col = "tab:red" if eddy_type == 1 else "tab:blue"
    if ms is not None:
        ax.plot(ms[:, 0], ms[:, 1], color=col, lw=1.5, transform=crs, zorder=4)
    if oc is not None:
        ax.plot(oc[:, 0], oc[:, 1], color="k", lw=0.8, ls=":", transform=crs, zorder=4)


def map_legend(ax, cfg, day, extra):
    ax.legend(handles=[
        mlines.Line2D([], [], color=eddy_color(cfg), lw=1.5, label=f"Max-speed contour {day:%m-%d}"),
        mlines.Line2D([], [], color="k", ls=":", lw=0.8, label="Outermost contour"),
        mlines.Line2D([], [], color="0.3", label="Ship track"),
        mlines.Line2D([], [], color="k", marker="x", ls="", label="TOEddies center"),
    ] + extra, fontsize=6, loc="lower left", framealpha=0.9)


def plot_profiles(ax, x, z, V, ylabel, title):
    """Panel (b): V(x) at PROFILE_DEPTHS."""
    cmap = plt.get_cmap("viridis")
    for k, zz in enumerate(PROFILE_DEPTHS):
        iz = np.argmin(np.abs(z - zz))
        if abs(z[iz] - zz) > 30:
            continue
        ax.plot(x, V[:, iz], "-o", ms=2.5, lw=1.2,
                color=cmap(k / max(1, len(PROFILE_DEPTHS) - 1)), label=f"{z[iz]:.0f} m")
    ax.axhline(0, color="k", lw=0.6)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7, ncol=len(PROFILE_DEPTHS), loc="best")
    ax.grid(alpha=0.3)
    ax.set_title(title, loc="left", fontsize=10)
    plt.setp(ax.get_xticklabels(), visible=False)


def plot_section(ax, x, z, V, vlim, xlabel, title, cblabel):
    """Panel (c): contoured V(x, z), colorbar outside so (b) and (c) stay aligned."""
    Vm = np.ma.masked_invalid(V.T)
    cf = ax.contourf(x, z, Vm, levels=np.linspace(-vlim, vlim, 21), cmap=CMAP, extend="both")
    try:
        cs = ax.contour(x, z, Vm, levels=np.arange(-2, 2.01, 0.1), colors="k", linewidths=0.4)
        ax.clabel(cs, fmt="%.1f", fontsize=6)
    except Exception:
        pass
    ax.set_ylim(np.nanmax(z), 0)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Depth (m)")
    ax.set_title(title, loc="left", fontsize=10)
    cb = plt.colorbar(cf, cax=ax.inset_axes([1.015, 0, 0.02, 1]))
    cb.set_label(cblabel)


def new_figure():
    fig = plt.figure(figsize=(18, 9))
    gs = GridSpec(2, 2, width_ratios=[1, 2.2], height_ratios=[1, 1.4], hspace=0.12, wspace=0.12,
                  figure=fig)
    axm = fig.add_subplot(gs[:, 0], projection=crs)
    axp = fig.add_subplot(gs[0, 1])
    axs = fig.add_subplot(gs[1, 1], sharex=axp)
    return fig, axm, axp, axs


# =============================================================================
# Cyclone: straight section through the eddy
# =============================================================================
def run_section(cfg, tr, out_dir):
    t0, t1 = cfg["t0"], cfg["t1"]
    tmid = pd.Timestamp(t0) + (pd.Timestamp(t1) - pd.Timestamp(t0)) / 2
    clon, clat = (float(a[0]) for a in center_at(tr, [tmid]))
    rmax = rmax_at(tr, t0, t1)
    col = eddy_color(cfg)

    sad = sadcp_window(t0, t1)
    lon0, lat0 = float(np.nanmean(sad["lon"])), float(np.nanmean(sad["lat"]))
    xs, ys = to_xy(sad["lon"], sad["lat"], lon0, lat0)
    origin, es, en = fit_line(xs, ys)
    proj_s = lambda x, y: (x - origin[0]) * es[0] + (y - origin[1]) * es[1]
    proj_n = lambda x, y: (x - origin[0]) * en[0] + (y - origin[1]) * en[1]
    cx, cy = to_xy(clon, clat, lon0, lat0)
    s_c, d_c = float(proj_s(cx, cy)), float(proj_n(cx, cy))
    heading = (np.degrees(np.arctan2(es[0], es[1])) + 360) % 360

    s = proj_s(xs, ys)
    vn = sad["u"] * en[0] + sad["v"] * en[1]
    xg, V, _ = bin_columns(s, vn, np.arange(np.floor(s.min()), np.ceil(s.max()) + DS_KM, DS_KM))
    z = sad["z"]
    V = fill_z_gaps(V, z)
    vlim = cfg.get("vlim") or vlim_of(V)

    fig, axm, axp, axs = new_figure()

    # (a) map
    pad = max(0.5, rmax / 111 * 1.3)
    extent = [min(np.nanmin(sad["lon"]), clon) - pad, max(np.nanmax(sad["lon"]), clon) + pad,
              min(np.nanmin(sad["lat"]), clat) - pad, max(np.nanmax(sad["lat"]), clat) + pad]
    geo_map(axm, extent, tr, tmid, cfg["eddy_type"])
    axm.plot(sad["lon"], sad["lat"], "-", color="0.3", lw=1, transform=crs, zorder=5)
    axm.scatter(sad["lon"], sad["lat"], s=4, c="k", transform=crs, zorder=6)
    smin, smax = np.nanmin(s), np.nanmax(s)
    llon, llat = to_lonlat(origin[0] + np.array([smin, smax]) * es[0],
                           origin[1] + np.array([smin, smax]) * es[1], lon0, lat0)
    tf = crs._as_mpl_transform(axm)
    axm.annotate("", xy=(llon[1], llat[1]), xytext=(llon[0], llat[0]), xycoords=tf, textcoords=tf,
                 arrowprops=dict(arrowstyle="->", color="m", lw=1.5), zorder=8)
    nlen = (smax - smin) * 0.15
    mx, my = llon.mean(), llat.mean()
    ax_, ay_ = to_lonlat(en[0] * nlen, en[1] * nlen, mx, my)
    axm.annotate("+v$_\\perp$", xy=(ax_, ay_), xytext=(mx, my), xycoords=tf, textcoords=tf,
                 arrowprops=dict(arrowstyle="->", color="m"), color="m", fontsize=8, zorder=8)
    axm.plot(clon, clat, "x", color="k", ms=8, mew=2, transform=crs, zorder=9)
    map_legend(axm, cfg, tmid, [mlines.Line2D([], [], color="m", label="Section axis")])
    axm.set_title(f"(a) {cfg['name']}\nline heading {heading:.0f}°, center {abs(d_c):.0f} km "
                  f"from the line", loc="left", fontsize=10)

    # (b) and (c), x = distance from the point of the line closest to the center
    x = xg - s_c

    def marks(ax):
        ax.axvline(0, color="k", ls="--", lw=1)
        if np.isfinite(rmax) and rmax > abs(d_c):
            for xx in (-1, 1):
                ax.axvline(xx * np.sqrt(rmax ** 2 - d_c ** 2), color=col, lw=1.5)

    plot_profiles(axp, x, z, V, "v$_\\perp$ (m s$^{-1}$)",
                  "(b) SADCP: velocity orthogonal to the section at selected depths")
    marks(axp)
    plot_section(axs, x, z, V, vlim,
                 f"Distance along the section from the center (km), + towards {heading:.0f}°",
                 f"(c) SADCP: v$_\\perp$, + to the left of travel (towards {(heading - 90) % 360:.0f}°)",
                 "v$_\\perp$ (m s$^{-1}$)")
    marks(axs)
    axs.set_xlim(np.nanmin(x), np.nanmax(x))
    axs.text(0, 0, " center", va="bottom", fontsize=7)

    fig.suptitle(f"{cfg['name']}: {pd.Timestamp(t0):%Y-%m-%d %H:%M} → {pd.Timestamp(t1):%Y-%m-%d %H:%M}"
                 f" | R$_{{max}}$ (TOEddies) {rmax:.0f} km | samples within "
                 f"{np.nanmax(np.abs(proj_n(xs, ys))):.0f} km of the section axis",
                 fontsize=12, y=0.96)
    out = out_dir / f"{cfg['name']}_SADCP_orthogonal_velocity.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {out}")


# =============================================================================
# Anticyclone: radial decomposition around the (moving) center
# =============================================================================
def run_radial(cfg, tr, out_dir):
    t0, t1 = cfg["t0"], cfg["t1"]
    rlim = cfg.get("r_limit_km", 200.0)
    rmax = rmax_at(tr, t0, t1)
    tmid = pd.Timestamp(t0) + (pd.Timestamp(t1) - pd.Timestamp(t0)) / 2
    col = eddy_color(cfg)

    sad = sadcp_window(t0, t1)
    clon_t, clat_t = center_at(tr, sad["time"])
    x = np.deg2rad(sad["lon"] - clon_t) * R_KM * np.cos(np.deg2rad(clat_t))
    y = np.deg2rad(sad["lat"] - clat_t) * R_KM

    dx = dy = 0.0
    if FIT_CENTER:
        lay = (sad["z"] >= FIT_LAYER[0]) & (sad["z"] <= FIT_LAYER[1])
        with np.errstate(all="ignore"):
            ul, vl = np.nanmean(sad["u"][:, lay], 1), np.nanmean(sad["v"][:, lay], 1)
        dx, dy, cost = fit_center(x, y, ul, vl, r_max=1.5 * rmax)
        print(f"  center refined by ({dx:+.0f}, {dy:+.0f}) km east/north of TOEddies "
              f"(radial energy fraction {cost:.2f})")
        if max(abs(dx), abs(dy)) >= FIT_SEARCH_KM - 1e-6:
            print(f"  ! the refined center is on the edge of the ±{FIT_SEARCH_KM:.0f} km search box:"
                  " the fit is not converged, check it or widen FIT_SEARCH_KM")
    x, y = x - dx, y - dy
    r, vt, _ = polar(x, y, sad["u"], sad["v"])
    rg, V, _ = bin_columns(r, vt, np.arange(0, rlim + R_BIN_KM, R_BIN_KM), nmin=3)
    z = sad["z"]
    V = fill_z_gaps(V, z)
    vlim = cfg.get("vlim") or vlim_of(V)
    sense = "anticlockwise" if np.nanmean(V[rg <= rmax]) > 0 else "clockwise"

    fig, axm, axp, axs = new_figure()

    # (a) map, around the refined center at mid-period
    clon, clat = (float(a[0]) for a in center_at(tr, [tmid]))
    rlon, rlat = to_lonlat(dx, dy, clon, clat)
    half = rlim / 111.0
    extent = [rlon - half / np.cos(np.deg2rad(rlat)), rlon + half / np.cos(np.deg2rad(rlat)),
              rlat - half, rlat + half]
    geo_map(axm, extent, tr, tmid, cfg["eddy_type"])
    used = r <= rlim
    axm.plot(sad["lon"], sad["lat"], "-", color="0.3", lw=0.8, transform=crs, zorder=5)
    axm.scatter(sad["lon"][used], sad["lat"][used], s=4, c="k", transform=crs, zorder=6)
    tt = pd.date_range(t0, t1, freq="6h")
    dlon, dlat = center_at(tr, tt)
    dlon, dlat = to_lonlat(dx, dy, dlon, dlat)
    axm.plot(dlon, dlat, color=col, ls="--", lw=1, transform=crs, zorder=7)
    axm.plot(clon, clat, "x", color="k", ms=8, mew=2, transform=crs, zorder=9)
    axm.plot(rlon, rlat, "+", color="k", ms=12, mew=2, transform=crs, zorder=9)
    map_legend(axm, cfg, tmid, [
        mlines.Line2D([], [], color="k", marker="+", ms=10, mew=2, ls="", label="Refined center"),
        mlines.Line2D([], [], color=col, ls="--", label="Refined center drift"),
        mlines.Line2D([], [], color="k", marker=".", ls="", label=f"SADCP used (r < {rlim:.0f} km)")])
    axm.set_title(f"(a) {cfg['name']}\ncenter refined by ({dx:+.0f}, {dy:+.0f}) km",
                  loc="left", fontsize=10)

    # (b) and (c)
    plot_profiles(axp, rg, z, V, "v$_\\theta$ (m s$^{-1}$)",
                  "(b) SADCP: azimuthal velocity vs radius at selected depths")
    axp.axvline(rmax, color=col, lw=1.5)
    plot_section(axs, rg, z, V, vlim, "Distance to the eddy center (km)",
                 f"(c) SADCP: v$_\\theta$(r, z), + anticlockwise (this eddy turns {sense})",
                 "v$_\\theta$ (m s$^{-1}$)")
    axs.axvline(rmax, color=col, lw=1.5)
    axs.text(rmax, 0, " R$_{max}$", va="bottom", fontsize=7)
    axs.set_xlim(0, rlim)

    fig.suptitle(f"{cfg['name']}: {pd.Timestamp(t0):%Y-%m-%d} → {pd.Timestamp(t1):%Y-%m-%d}, frame "
                 f"moving with the eddy | R$_{{max}}$ (TOEddies) {rmax:.0f} km", fontsize=12, y=0.96)
    out = out_dir / f"{cfg['name']}_SADCP_azimuthal_velocity.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {out}")


# =============================================================================
if __name__ == "__main__":
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    for key in RUN:
        cfg = EDDIES[key]
        print(f"\n=== {key} ({cfg['name']}): {cfg['mode']} {cfg['t0']} -> {cfg['t1']}")
        tr = load_track(cfg["name"])
        (run_section if cfg["mode"] == "section" else run_radial)(cfg, tr, OUT_DIR)


=== CE18 (CE18_20260523): section 2026-07-16T06:00 -> 2026-07-17T16:00
  WHIRLS_NB150_TOT_osite.nc: using UVEL_ADCP_CORTIDE, VVEL_ADCP_CORTIDE
  WHIRLS_BB38_TOT_osite.nc: using UVEL_ADCP_CORTIDE, VVEL_ADCP_CORTIDE
  saved /Users/emmagurcan/Documents/France/ENS/Speich/WHIRLS/figures2/CE18_20260523_SADCP_orthogonal_velocity.png

=== AE11 (AE11_20260514): radial 2026-07-01T00:00 -> 2026-07-11T00:00
  center refined by (-60, -8) km east/north of TOEddies (radial energy fraction 0.04)
  ! the refined center is on the edge of the ±60 km search box: the fit is not converged, check it or widen FIT_SEARCH_KM
  saved /Users/emmagurcan/Documents/France/ENS/Speich/WHIRLS/figures2/AE11_20260514_SADCP_azimuthal_velocity.png
